En esta etapa de gold tomamos la tabla de transacciones ya limpia y libre de duplicados y generamos tablas factuales y dimensionales

- Clientes (Dim)
- Cotizaciones (Dim)
- Instrumentos (Dim)

- fact_transaction: Transacciones enriquecidas sin agrupaciones
- fact_transaction_daily: Transacciones con granularidad por dia por cada cliente

In [0]:
%sql
SELECT * FROM iol_challenge.silver.deduped_transactions LIMIT 50;

sk_transaccion,fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen,dia,anio,mes,timestamp_ejecucion,errores_calidad,tiene_errores_calidad
654c56453d1900ac678ac8e27d886e30,2026-01-28T06:47:26.000Z,Venta,CLI323A59A0,"Cedear Netflix, Inc.",ARS,NFLX,3,2691.1216,TXN00130gbuq5cb8,App Mobile,1,2026,1,2026-08-05T05:58:01.342Z,null,false
0527e45976cf28772e0e0818c1b9d49f,2026-02-05T01:34:05.000Z,Venta,CLI0B0868D7,Grupo Financiero Galicia S.A,ARS,GGAL,6,7995.8703,TXN0014pi30pgnh1,App Mobile,2,2026,2,2026-08-05T05:58:01.342Z,null,false
c34448d3914a9d07172aa0969b952ba8,2026-01-30T09:42:23.000Z,Compra,CLIFCE7A53B,Bbva,ARS,BBAR,1,9972.4962,TXN001671nd5iug5,App Mobile,1,2026,1,2026-08-05T05:58:01.342Z,null,false
40adf9a6bb50b043ce76ea5fa1651a92,2026-03-02T19:45:51.000Z,Compra,CLI42C89E5A,Transportadora Gas del Sur,ARS,TGSU2,3,9152.431,TXN001d63us6ohw0,Sitio Web Desktop,3,2026,3,2026-08-05T05:58:01.342Z,null,false
c85d275546bddb8a2d393196a51bcefa,2026-02-02T21:48:01.000Z,Compra,CLIA02ABE02,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,2,884.6185,TXN001whi1fl1omy,App Mobile,2,2026,2,2026-08-05T05:58:01.342Z,null,false
79e2af93f8bc7dadf331ec00fc0706e5,2026-02-09T19:08:49.000Z,Compra,CLI1F89EF2D,Cedear The Coca Cola Company,ARS,KO,2,23505.5037,TXN0023q8749xe1v,App Mobile,2,2026,2,2026-08-05T05:58:01.342Z,null,false
b7523bec29ae3f89ffb74736533cac71,2026-03-13T05:02:21.000Z,Venta,CLI42F444C8,Cedear Spdr S&P 500,ARS,SPY,2,49040.4264,TXN0028manmlvlj5,App Mobile,3,2026,3,2026-08-05T05:58:01.342Z,null,false
aa1826316d7c6f0d261711b36418c7f6,2026-01-14T19:35:22.000Z,Compra,CLI36A35DBC,Cedear Microsoft Corp.,ARS,MSFT,2,23521.2603,TXN00395gzt3a24z,Sitio Web Responsive,1,2026,1,2026-08-05T05:58:01.342Z,null,false
0a1dd4eeb2ba62a88704f65fa14afd3d,2026-01-13T19:51:18.000Z,Compra,CLIF0893C5A,Cedear Taiwan Semic. Manuf.,ARS,TSM,1,56512.0619,TXN004ov11xzrlff,App Mobile,1,2026,1,2026-08-05T05:58:01.342Z,null,false
a30a444864a743d0a93a295028eb3eb5,2026-02-04T02:50:41.000Z,Venta,CLI3FC35AB3,Aluar,ARS,ALUA,10,980.8941,TXN004ru35okfz0t,Sitio Web Desktop,2,2026,2,2026-08-05T05:58:01.342Z,null,false


In [0]:
%sql
CREATE OR REPLACE TABLE iol_challenge.gold.dim_clientes AS (
    SELECT 
        MD5(CONCAT_WS('||', id_cliente)) AS sk_cliente
        ,id_cliente
        ,MIN(fecha) AS fecha_primera_transaccion
        ,MAX(fecha) AS fecha_ultima_transaccion
        ,MIN_BY(sk_transaccion, fecha) AS sk_primera_transaccion
        ,MAX_BY(sk_transaccion, fecha) AS sk_ultima_transaccion
        ,SUM(CASE WHEN tipoTran = 'Compra' THEN 1 ELSE 0 END) AS total_compras
        ,SUM(CASE WHEN tipoTran = 'Venta' THEN 1 ELSE 0 END)  AS total_ventas
        ,COUNT(*) AS total_transacciones
        FROM 
            iol_challenge.silver.deduped_transactions
            GROUP BY id_cliente
)

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM iol_challenge.gold.dim_clientes LIMIT 10;

sk_cliente,id_cliente,fecha_primera_transaccion,fecha_ultima_transaccion,sk_primera_transaccion,sk_ultima_transaccion,total_compras,total_ventas,total_transacciones
689156e652614a70399a75ac23e630b6,CLI42F444C8,2026-03-13T05:02:21.000Z,2026-03-13T05:02:21.000Z,b7523bec29ae3f89ffb74736533cac71,b7523bec29ae3f89ffb74736533cac71,0,1,1
e9b4ec963445f2916499430db90c492f,CLI2CA23B71,2026-01-09T23:52:53.000Z,2026-01-09T23:52:53.000Z,02b1a575cea157dcd8f81dafabcb7d9b,02b1a575cea157dcd8f81dafabcb7d9b,1,0,1
1fa4e3c73c4be61070def63b8e67b10b,CLI2AEC73C3,2026-01-26T22:36:07.000Z,2026-01-26T22:36:07.000Z,4c680fd26d3240c709271399d5d30edc,4c680fd26d3240c709271399d5d30edc,1,0,1
50ee338569c00146334694b6357f0d6a,CLI87F49ED2,2026-01-29T21:00:26.000Z,2026-01-29T21:00:26.000Z,bf9ec0f9f67a6fca0f04183ef7bb9d4d,bf9ec0f9f67a6fca0f04183ef7bb9d4d,1,0,1
c1c9e4e80126b89dbb08992fb1118f54,CLIB57A9D97,2026-01-20T20:33:41.000Z,2026-01-20T20:33:41.000Z,d60e9d22877f5a25ffd074607dc205cb,d60e9d22877f5a25ffd074607dc205cb,1,0,1
170d7bf317b4a231aa6dfea93fa50218,CLI11354CDA,2026-01-13T18:27:40.000Z,2026-03-12T07:11:27.000Z,0392c048c8ca3263459e7338336ac8ed,00a1d9179827a8ec65183130084434b9,4,3,7
c10e23f009f8315413294fefd66747a5,CLI01C41343,2026-01-02T14:50:37.000Z,2026-03-11T06:31:49.000Z,63bbb9c5b1987d9532c45963d97473b8,883bdb3d75a88b6a8c2c536040789a5d,26,10,36
eb34ec0cd57011695645f980c3a58b50,CLIE0031807,2026-01-14T13:48:40.000Z,2026-01-30T06:13:35.000Z,4f2da6992965545d0fd04a4b8cbc474d,07435d00bac77f5084b3e992030a8f0c,6,5,11
898a4eb087da96dab57a58fd15e6ca55,CLI6AA73FA2,2026-01-23T20:49:53.000Z,2026-03-12T12:13:46.000Z,c8453b62f3d84ebb14c940583134ae48,ec10b529a8471dd9abe627621b1e0552,3,2,5
0360f438691ccc7615ceda4c9e8dd913,CLICF34FD44,2026-01-17T17:29:54.000Z,2026-03-06T09:44:25.000Z,fc04830bb510264f2e6d72d671537ad1,ee149a2b6545d65765e8d073dd0da2f5,3,0,3


Generamos tabla gold con las cotizaciones por simbolo.

Scope: Tomamos cómo criterio listas fijas de cada simbolo para la clasificación. La solución ideal sería usar una API externa para completar la clasificación completa de los instrumentos, lo cuál permitiria análizar de manera completa todo el conjunto de datos.

In [0]:
%sql
CREATE OR REPLACE TABLE iol_challenge.gold.dim_cotizaciones AS
    (
        SELECT 
            MD5(CONCAT_WS('||', simbolo_titulo)) AS sk_simbolo
            ,simbolo_titulo
            ,CASE 
                WHEN
                    simbolo_titulo IN (
                        -- Ley Argentina (Bonares / Ley Local)
                        "AL29", "AL30", "AL35", "AE38", "AL41", "AN29",
                        "AL29D", "AL30D", "AL35D", "AE38D", "AL41D", "AN29D",
                        "AL29C", "AL30C", "AL35C", "AE38C", "AL41C", "AN29C",

                        -- Ley Nueva York (Globales / Ley Extranjera)
                        "GD29", "GD30", "GD35", "GD38", "GD41", "GD46",
                        "GD29D", "GD30D", "GD35D", "GD38D", "GD41D", "GD46D",
                        "GD29C", "GD30C", "GD35C", "GD38C", "GD41C", "GD46C"
                    )
                    THEN "Bono soberano"
                WHEN
                    simbolo_titulo IN (
                        "AAL", "AAPL", "ABBV", "ABT", "ACN", "ADB", "ADE", "ADI", "ADP", "AEM", 
                        "AIG", "AMAT", "AMD", "AMX", "AMZN", "ARCO", "ARKG", "ASML", "AUY", "AVGO", 
                        "BA", "BAC", "BABA", "BBD", "BBV", "BCS", "BIIB", "BITF", "BMA", "BMY", 
                        "BRKB", "C", "CAT", "CBD", "CDE", "CED", "CL", "COIN", "COST", "CRM", 
                        "CSCO", "CVX", "CX", "DE", "DEO", "DIA", "DIS", "EBAY", "EEM", "EFX", 
                        "ERJ", "ETSY", "EWZ", "F", "FCX", "FDX", "FSLR", "GE", "GFI", "GGB", 
                        "GILD", "GLOB", "GM", "GOOGL", "GS", "HAL", "HMC", "HMY", "HOM", "HON", 
                        "HPQ", "HSBC", "IBM", "IFF", "INFY", "INTC", "IP", "IWM", "JNJ", "JPM", 
                        "KMB", "KO", "LFC", "LLY", "LMT", "LRCX", "MA", "MCD", "MELI", "META", 
                        "MMM", "MO", "MRK", "MSFT", "MU", "NEM", "NFLX", "NKE", "NTES", "NVDA", 
                        "ORCL", "PBR", "PEP", "PFE", "PG", "PYPL", "QCOM", "QQQ", "RIO", "ROST", 
                        "SBUX", "SCCO", "SHEL", "SHOP", "SLB", "SNAP", "SNOW", "SPOT", "SPY", "SQ", 
                        "STNE", "T", "TARGET", "TEFO", "TGT", "TM", "TSLA", "TSM", "TX", "TXN", 
                        "UNH", "UNP", "UPST", "V", "VALE", "VIST", "VOD", "VZ", "WBA", "WFC", 
                        "WMT", "X", "XOM", "XP", "YMM", "ZM"
                    ) THEN 'Cedear'
                WHEN 
                    simbolo_titulo IN (
                        "AGRO", "ALUA", "AUSO", "BBAR", "BHIP", "BMA", "BPAT", "BRIO", "BYMA", 
                        "CAPX", "CARC", "CECO2", "CELU", "CEPU", "CGPA2", "COME", "CRES", "CTIO", 
                        "CVH", "DGCU2", "DOME", "DYCA", "EDN", "FERR", "FIPL", "GALA", "GARO", 
                        "GBAN", "GCDI", "GGAL", "GRIM", "HARG", "INTR", "INVJ", "IRS2", "LOMA", 
                        "METR", "MILI", "MOLI", "MORI", "MTR", "OPAR", "PAMP", "PATA", "PGPRI", 
                        "SOMI", "SUPV", "TECO2", "TGNO4", "TGSU2", "TRAN", "TXAR", "VALO", "YPFD"
                    ) THEN 'Accion local'
                ELSE
                    'A revisar'
            END
            AS simbolo_tipo
            ,MIN_BY(sk_transaccion, fecha) AS primera_transaccion_sk
            ,MAX_BY(sk_transaccion, fecha) AS ultima_transaccion_sk
            ,min(FECHA) AS primera_transaccion_fecha
            ,max(fecha) AS ultima_transaccion_fecha
            ,MIN_BY(c.sk_cliente, fecha) AS primera_transaccion_cliente_sk
            ,MAX_BY(c.sk_cliente, fecha) AS ultima_transaccion_cliente_sk
            ,SUM(CASE WHEN tipoTran = 'Venta' THEN 1 ELSE 0 END) AS total_compras
            ,SUM(CASE WHEN tipoTran = 'Compra' THEN 1 ELSE 0 END) AS total_ventas
            ,FIRST(moneda) AS moneda
            ,COUNT(*) AS total_transacciones
            FROM iol_challenge.silver.deduped_transactions d
                LEFT JOIN iol_challenge.gold.dim_clientes c
                    ON c.id_cliente = d.id_cliente
                GROUP BY simbolo_titulo
    )

num_affected_rows,num_inserted_rows


Observamos qué hay muchos casos qué las listas fijas no lograron clasificar

In [0]:
%sql
SELECT simbolo_tipo, count(*) FROM iol_challenge.gold.dim_cotizaciones GROUP BY simbolo_tipo;

simbolo_tipo,count(*)
Cedear,128
Accion local,43
A revisar,1243
Bono soberano,31


Construimos una tabla factual con todas las transacciones

In [0]:
%sql
CREATE OR REPLACE TABLE iol_challenge.gold.fact_transaction AS (
    SELECT 
        t.id_cliente
        ,t.moneda
        ,t.precio
        ,t.cantidad
        ,t.origen

        ,t.fecha
        ,t.simbolo_titulo
        ,c.simbolo_tipo
    
        ,di.High
        ,di.Low
        ,di.Open
        ,di.Close
        ,di.Volume
        ,(di.High + di.Low) / 2 as valor_mercado_promedio

    FROM iol_challenge.silver.deduped_transactions t
        LEFT JOIN iol_challenge.gold.dim_cotizaciones c
            ON c.simbolo_titulo = t.simbolo_titulo
        LEFT JOIN iol_challenge.silver.deduped_instruments di
            ON t.simbolo_titulo = di.simbolo AND di.date <= t.fecha
            --- Tomamos la ultima fecha disponible con información de cotización qué sea anterior a la fecha correspondiente a la agregación
            QUALIFY ROW_NUMBER() OVER (PARTITION BY t.fecha, t.id_cliente, t.simbolo_titulo, t.tipoTran ORDER BY di.date DESC) = 1 
)

num_affected_rows,num_inserted_rows


Construimos una tabla factual con las transacciones por fecha, por cliente, instrumento e origen, enriqueciendo con las información de cotizaciones diarias. En primera instancia se agrupa el volumen transaccionado en el periodo de tiempo y cliente en cuestión pero se pueden agregar más campos.

In [0]:
%sql
CREATE OR REPLACE TABLE iol_challenge.gold.fact_transaction_daily AS (
WITH grouped_data AS (
    SELECT
        t.fecha::date
        ,t.id_cliente
        ,t.simbolo_titulo
        ,t.tipoTran
        ,t.origen
        ,SUM(CASE WHEN t.moneda='ARS' THEN t.cantidad * t.precio ELSE 0 END) AS volumen_transaccionado_ars
        ,SUM(CASE WHEN t.moneda='USD' THEN t.cantidad * t.precio ELSE 0 END) AS volumen_transaccionado_usd
        ,COUNT(*) as total_transactions
            FROM iol_challenge.silver.deduped_transactions t
        GROUP BY t.fecha::date, t.id_cliente, t.simbolo_titulo, t.tipoTran, t.origen
    )
    SELECT
        -- Información sobre la cotización del instrumento
        g.fecha
        ,g.id_cliente
        ,g.simbolo_titulo
        ,g.tipoTran
        ,g.volumen_transaccionado_ars
        ,g.volumen_transaccionado_usd
        ,g.origen
        ,g.total_transactions

        ,di.High
        ,di.Low
        ,di.Open
        ,di.Close
        ,di.Volume
        ,(di.High + di.Low) / 2 as valor_mercado_promedio
        
            FROM grouped_data g
                LEFT JOIN iol_challenge.silver.deduped_instruments di
                    ON g.simbolo_titulo = di.simbolo AND di.date <= g.fecha
            --- Tomamos la ultima fecha disponible con información de cotización qué sea anterior a la fecha correspondiente a la agregación
            QUALIFY ROW_NUMBER() OVER (PARTITION BY g.fecha, g.id_cliente, g.simbolo_titulo, g.tipoTran ORDER BY di.date DESC) = 1
)

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM iol_challenge.gold.fact_transaction_daily limit 10;

fecha,id_cliente,simbolo_titulo,tipoTran,volumen_transaccionado_ars,volumen_transaccionado_usd,origen,total_transactions,High,Low,Open,Close,Volume,valor_mercado_promedio
2026-01-02,CLI003B5F22,SPY,Compra,105075.9544,0.0,App Mobile,1,17594.76587540064,17403.42864582942,17569.80876068376,17519.89453125,378513.0,17499.09726061503
2026-01-02,CLI007C80B0,AMZN,Compra,4879.6584,0.0,App Mobile,1,2510.0,2405.0,2475.0,2426.0,986592.0,2457.5
2026-01-02,CLI008D7ED2,AL30D,Venta,0.0,104.625,App Mobile,1,null,null,null,null,null,null
2026-01-02,CLI00E0116F,ALUA,Compra,998.5252,0.0,App Mobile,1,1009.0,975.0,1000.0,1004.0,388923.0,992.0
2026-01-02,CLI00FF7AE6,GFGC85539F,Compra,44978.0786,0.0,Sitio Web Desktop,1,null,null,null,null,null,null
2026-01-02,CLI013E0D2E,TECO2,Compra,47142.9608,0.0,App Mobile,1,3625.0,3500.0,3530.0,3595.0,152398.0,3562.5
2026-01-02,CLI018D651B,PATH,Venta,540933.012,0.0,Sitio Web Responsive,1,12800.0,11940.0,12700.0,12400.0,27291.0,12370.0
2026-01-02,CLI018FA06B,AL30,Compra,49196.627199999995,0.0,App Mobile,1,null,null,null,null,null,null
2026-01-02,CLI019A6CDE,AL30,Compra,15950.5968,0.0,App Mobile,1,null,null,null,null,null,null
2026-01-02,CLI01C41343,AMZN,Compra,55669.0574,0.0,App Mobile,1,2510.0,2405.0,2475.0,2426.0,986592.0,2457.5
